# How to Create an Artificial UFD Candidate

This tutorial will walk you through the process of generating an artificial candidate. At the end of the tutorial, there is a section titled "Other helpful tips" that provides details on training a GMM from scratch, and creating multiple galaxies with random parameters.

### 0. Import Necessary Packages

In [ ]:
from myplotstyle import plt
from astropy import units as u

from artificial_galaxy_pipeline import ArtificialGalaxy, BackgroundImage, ImageInjector, artificial_galaxy_generator
from artificial_galaxy_pipeline.gmm import GMM, GMMBuilder
from artificial_galaxy_pipeline.viz import SourceData

### 1. Generate the injected stellar sources

We’ll will start by specifying the parameters for the galaxy we want to create, and passing them into the <small>`ArtificialGalaxy`</small> class. This class generates the magnitude and posititon data for the artificial stellar sources based on the provided parameteres.

Here we use an example of known [Leo K dwarf galaxy](https://iopscience.iop.org/article/10.3847/1538-4357/ad429b).

In [ ]:
leo_k = ArtificialGalaxy(
    name="Leo_K",
    log_age=10.08,
    feh=-1.88,
    total_mass=1.9e4 * u.Msun,
    scale_radius=71.54 * u.pc,
    distance=0.43 * u.Mpc,
)

# This shows the properties of the artificial stars in the galaxy
leo_k.source_table[:10] 

### 2. Obtain the Background 

We then need to obtain the real sky background we plan to inject into. We will create an instance of  <small>`BackgroundImage`</small> class.

In [ ]:
background = BackgroundImage(ra=182.5002, dec=12.5554)

fig = plt.figure(dpi=80)
plt.imshow(background.rgb)
plt.axis('off')

### 3. Inject the stellar sources into the background image

We will pass in our artifical galaxy and background to <small>`ImageInjector`</small> which handles the injection process.

In [ ]:
injector = ImageInjector(leo_k, background)

fig = plt.figure(dpi=80)
plt.imshow(injector.rgb)
plt.axis('off')

### 4. Obtain the pre-trained GMM

Before we create the diagnostic plot we need to load in the pre-trained GMM. The GMM is trained on 200 images of artificial UFDs. 

You can download pre-trained GMM file [here](https://github.com/alexb000/artificial-galaxy-pipeline/blob/main/trained_gmm.pkl).

**Note:** If you are interested in training a GMM on new data see section "Training a GMM" at the bottom of the tutorial.


In [ ]:
gmm = GMM(model_path="trained_gmm.pkl")      # Returns the pre-trained GMM

### 5. : Create the Diagnostic Plot

Once you have an instance of <small>`ImageInjector`</small>, and <small>`BackgroundImage`</small> you can access both the observed magnitude table and the RGB image of your injected galaxy.  With this information, we can create an instance of the <small>`SourceData`</small> class, which generates a diagnostic plot similar to the multi-panel figures shown in the research note (*link*).  

This diagnostic plot includes:
- density plot
- color–magnitude diagram (CMDs)
- hexbin plot

In [ ]:
# obtain the observed magnitude table for the injected sources
observed_source_table = leo_k.get_observed_source_table(gmm, background.ra, background.dec) 

# obtain the injected image
rgb = injector.rgb 

source = SourceData(ra=background.ra,
                    dec=background.dec,
                    source_inj_table=observed_source_table,
                    inj_image=rgb)
source.make_plot_minimal()


# Other Helpful Tips

### 1. How to create a UFD Candidate(s) with random parameters
We can create two types of galaxy candidates:  
- Those with *specified feature parameters* (shown in step 1 of the tutorial above), or  
- Those with *random feature parameters* drawn from uniform distributions that match known ultra-faint dwarfs (UFDs).  The code below shows you have to create
galaxies with randomly generated parameters.


In [ ]:
for galaxy in artificial_galaxy_generator(num_sources=5):
    injector = ImageInjector(galaxy, background)

    # Save the image and source table to file
    injector.write_image_to_file(f"{galaxy[name]}.fits")
    injector.write_source_table(f"{galaxy[name]}.parquet")

    fig = plt.figure(dpi=80)
    plt.imshow(injector.rgb)
    plt.axis('off')
    plt.show()
    plt.close()
    

### 2. How to train a GMM
There are two ways to train a new GMM...

#### 2.1 Automatically generate new data to train a GMM

If you want to train a new GMM using freshly generated data, you can let <small>`GMMBuilder`</small> automatically create everything for you.

When initialized without specifying training or test names, <small>`GMMBuilder`</small> will:

- Generate (by default) 200 artificial UFDs for training and 100 artificial UFDs for testing,
- Create an instance of <small>`MagnitudeExtractor`</small> for each to image to preform source extraction and photometry.
- Train a new GMM using the extracted magnitudes


In [ ]:
builder = GMMBuilder()
gmm = builder.gmm

#### 2.2: Use existing artificial UFDs to train a GMM

If you already have FITS images and magnitude files from the injection process, you can pass in explicit galaxy names to <small>`GMMBuilder`</small>.

When initialized, <small>`GMMBuilder`</small> will automatically:

- Preform source extraction and photometry on the corresponding training and test files,
- Train the GMM using the extracted magnitudes.

In [ ]:
galaxy_names_train = [f"galaxy_train_{i:04d}" for i in range(200)]
galaxy_names_test  = [f"galaxy_test_{i:04d}" for i in range(100)]

builder = GMMBuilder(train_names=galaxy_names_train, test_names=galaxy_names_test, data_folder="gmm_training_data", model_path="new_trained_gmm.pkl")
gmm = builder.gmm